In [55]:
# from DataHandler.Vocab import CharTokenizer
from NanoGPT import NanoGPT as gpt
import torch
from tokenizers import Tokenizer

In [20]:
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

<!-- this is for early traning to make -->

In [ ]:


train_text = open("DataHandler/txts/train.txt", encoding="utf-8").read()
val_text   = open("DataHandler/txts/val.txt", encoding="utf-8").read()

train_ids = torch.tensor(tokenizer.encode(train_text).ids, dtype=torch.long)
val_ids   = torch.tensor(tokenizer.encode(val_text).ids, dtype=torch.long)

In [21]:
block_size = 128
batch_size = 64

In [22]:
def get_batch(data):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    
    return x, y

In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = gpt(
    vocab_size=4000,
    block_size=256,
    n_embd=384,
    n_head=6,
    n_layer=6
).to(device)

In [15]:
model.load("NanoGPT.pt")

In [24]:

def AutoSaveBestOne(best_val_loss,current_val_loss,step):
    if current_val_loss < best_val_loss:
        torch.save(model.state_dict(), f"Models/NanoGPT_{step}.pt")
        print(f"model saved validation loss decreses from {best_val_loss:.4f} -> {current_val_loss:.4f}")
        return current_val_loss
    else:
        return best_val_loss

In [ ]:
max_iters = 20000
eval_interval = 500
eval_iters = 50  
best_val_loss = float("inf")
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max_iters
)

for step in range(1,max_iters+1):
    model.train()
    xb, yb = get_batch(train_ids)
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if step % eval_interval == 0:
        model.eval()

        val_losses = []

        with torch.no_grad():
            for _ in range(eval_iters):
                xb, yb = get_batch(val_ids)
                xb, yb = xb.to(device), yb.to(device)

                _, val_loss = model(xb, yb)
                val_losses.append(val_loss.item())

        avg_val_loss = sum(val_losses) / len(val_losses)

        print(f"Step {step} | Train Loss {loss.item():.4f} | Val Loss {avg_val_loss:.4f}")
        best_val_loss=AutoSaveBestOne(best_val_loss=best_val_loss,current_val_loss=avg_val_loss,step=step)

Real chat bot traning / finetuning begins here

<!-- its for last fine tuning for one line answer and the data is too tiny so it will overfit it and its fine for this so dont train over 50 iterations-->

In [50]:
from tokenizers import Tokenizer
import torch
train_text = open("DataHandler/short_chat_data/train.txt", encoding="utf-8").read()
val_text   = open("DataHandler/short_chat_data/val.txt", encoding="utf-8").read()

train_ids = torch.tensor(tokenizer.encode(train_text).ids, dtype=torch.long)
val_ids   = torch.tensor(tokenizer.encode(val_text).ids, dtype=torch.long)

<!-- its for AI to understand how chats structure are their so it can reply like human not just predict next token -->

In [54]:
from tokenizers import Tokenizer
import torch
train_text = open("DataHandler/chats/train.txt", encoding="utf-8").read()
val_text   = open("DataHandler/chats/val.txt", encoding="utf-8").read()

train_ids = torch.tensor(tokenizer.encode(train_text).ids, dtype=torch.long)
val_ids   = torch.tensor(tokenizer.encode(val_text).ids, dtype=torch.long)

In [ ]:
max_iters = 10
eval_interval = 1
eval_iters = 5
best_val_loss = float("inf")
model.load("Models/NanoGPT_9.pt")
optimizer = torch.optim.AdamW(model.parameters(),lr=5e-5,betas=(0.9, 0.95),weight_decay=0.01)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max_iters)

for step in range(1,max_iters+1):

    model.train()
    xb, yb = get_batch(train_ids)
    xb, yb = xb.to(device), yb.to(device)

    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    if step % eval_interval == 0:
        model.eval()

        val_losses = []

        with torch.no_grad():
            for _ in range(eval_iters):
                xb, yb = get_batch(val_ids)
                xb, yb = xb.to(device), yb.to(device)
                _, val_loss = model(xb, yb)
                val_losses.append(val_loss.item())

        avg_val_loss = sum(val_losses) / len(val_losses)

        print(f"Step {step} | Train Loss {loss.item():.4f} | Val Loss {avg_val_loss:.4f}")

        best_val_loss = AutoSaveBestOne(best_val_loss=best_val_loss,current_val_loss=avg_val_loss,step=step)

Step 1 | Train Loss 1.3923 | Val Loss 0.7208
model saved validation loss decreses from inf -> 0.7208
Step 2 | Train Loss 0.8684 | Val Loss 0.4776
model saved validation loss decreses from 0.7208 -> 0.4776
Step 3 | Train Loss 0.6371 | Val Loss 0.3822
model saved validation loss decreses from 0.4776 -> 0.3822
Step 4 | Train Loss 0.4842 | Val Loss 0.3271
model saved validation loss decreses from 0.3822 -> 0.3271
Step 5 | Train Loss 0.4201 | Val Loss 0.2931
model saved validation loss decreses from 0.3271 -> 0.2931
Step 6 | Train Loss 0.3787 | Val Loss 0.2784
model saved validation loss decreses from 0.2931 -> 0.2784
Step 7 | Train Loss 0.3543 | Val Loss 0.2634
model saved validation loss decreses from 0.2784 -> 0.2634
Step 8 | Train Loss 0.3340 | Val Loss 0.2595
model saved validation loss decreses from 0.2634 -> 0.2595
Step 9 | Train Loss 0.3231 | Val Loss 0.2552
model saved validation loss decreses from 0.2595 -> 0.2552
Step 10 | Train Loss 0.3187 | Val Loss 0.2559


In [9]:
model.eval()
with torch.no_grad():
    xb, yb = get_batch(val_ids)
    xb, yb = xb.to(device), yb.to(device)
    _, loss = model(xb, yb)
    print("Initial loss:", loss.item())

Initial loss: 3.856405735015869


In [53]:
from NanoGPT import generate
def test():
    model.load("Models/NanoGPT_9.pt")
    model.to(device)
    model.eval()

    prompt = "<USER>\nkirshn ji\n<AI>\n"

    encoded = tokenizer.encode(prompt).ids
    start = torch.tensor([encoded], dtype=torch.long, device=device)

    end_token_id = tokenizer.token_to_id("<END>")

    generated = generate(
        model,
        start,
        max_new_tokens=200,
        stop_token_id=end_token_id
    )

    output = tokenizer.decode(generated[0].tolist())
    ai_part = output.split("<AI>")[-1].replace("<END>", "").strip()

    print(ai_part)
test()

kirshn ji
 
Now to learn coding
